# 03 — Cluster Profiling & Churn Model

**Purpose:** Turn cluster numbers into business stories, evaluate both ML models.  
**Questions we answer here:**
- What defines each segment? What are their characteristics?
- Was k=5 the right choice? (Elbow method validation)
- Where is the revenue concentrated? What's at risk?
- How well does the churn model perform? What's the cost of inaction?

**Tables used:** `marts.mart_cluster_output`, `marts.mart_cluster_profiles`, `marts.mart_cluster_summary`, `marts.mart_churn_risk`  
**Models used:** `analytics.kmeans_customer_segments`, `analytics.churn_classifier`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

---
## Part A: K-Means Segmentation
---

### A1. Model evaluation — Davies-Bouldin index
Lower is better. Under 2.0 is good for business use.

In [ ]:
eval_result = q(f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT}.analytics.kmeans_customer_segments`)")
print(f"Davies-Bouldin Index: {eval_result.iloc[0]['davies_bouldin_index']:.4f}")
print(f"Mean Squared Distance: {eval_result.iloc[0]['mean_squared_distance']:.4f}")
eval_result

### A2. Training convergence
Loss should decrease and flatten — that means the model found stable cluster centers.

In [ ]:
training = q(f"""
    SELECT iteration,
           ROUND(loss, 4) AS loss,
           ROUND(loss - LAG(loss) OVER (ORDER BY iteration), 4) AS improvement
    FROM ML.TRAINING_INFO(MODEL `{PROJECT}.analytics.kmeans_customer_segments`)
    ORDER BY iteration
""")

fig = px.line(training, x='iteration', y='loss',
              title='Training convergence — loss should flatten',
              markers=True)
fig.show()
training

### A3. Elbow method — validating k=5
Train models for k=3 through k=8 and compare Davies-Bouldin scores.

**Note:** This section takes ~5 minutes to run (6 model trainings). Skip if already validated.

In [ ]:
FEATURE_SELECT = f"""
    SELECT val_trns, nr_trns, lst_trns_days, avg_val,
           active_destinations, active_nav_categories,
           NR_TRNS_WEEKEND, NR_TRNS_WEEK, active_months
    FROM `{PROJECT}.analytics.int_rfm_scores`
"""

results = []
for k in [3, 4, 5, 6, 7, 8]:
    model_name = f'{PROJECT}.analytics.elbow_k{k}' if k != 5 else f'{PROJECT}.analytics.kmeans_customer_segments'
    
    if k != 5:
        print(f'Training k={k}...')
        client.query(f"""
            CREATE OR REPLACE MODEL `{model_name}`
            OPTIONS (model_type='KMEANS', num_clusters={k},
                     standardize_features=TRUE, max_iterations=20)
            AS {FEATURE_SELECT}
        """).result()
    
    ev = q(f"SELECT {k} AS k, * FROM ML.EVALUATE(MODEL `{model_name}`)")
    results.append(ev)
    print(f'  k={k}: Davies-Bouldin = {ev.iloc[0]["davies_bouldin_index"]:.4f}')

elbow = pd.concat(results)
elbow

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=elbow['k'], y=elbow['davies_bouldin_index'],
                         mode='lines+markers', name='Davies-Bouldin',
                         line=dict(color='#2E75B6')))
fig.add_vline(x=5, line_dash='dash', line_color='red',
              annotation_text='k=5 (selected)')
fig.update_layout(title='Elbow method — Davies-Bouldin index by k',
                  xaxis_title='Number of clusters (k)',
                  yaxis_title='Davies-Bouldin index (lower = better)')
fig.show()

In [ ]:
# cleanup elbow models
for k in [3, 4, 6, 7, 8]:
    client.query(f'DROP MODEL IF EXISTS `{PROJECT}.analytics.elbow_k{k}`').result()
    print(f'Dropped elbow_k{k}')

### A4. Segment overview

In [ ]:
summary = q(f"""
    SELECT segment_name, customer_count, pct_of_total,
           avg_total_spend, avg_transactions, avg_recency_days,
           business_description, recommended_action
    FROM `{PROJECT}.marts.mart_cluster_summary`
    ORDER BY avg_total_spend DESC
""")
summary

### A5. Cluster centroids — what defines each group?

In [ ]:
centroids = q(f"""
    SELECT centroid_id, feature, ROUND(numerical_value, 2) AS value
    FROM ML.CENTROIDS(MODEL `{PROJECT}.analytics.kmeans_customer_segments`)
    ORDER BY centroid_id, feature
""")
centroid_pivot = centroids.pivot(index='feature', columns='centroid_id', values='value')
centroid_pivot

### A6. Indexed radar chart
Each metric indexed to 0-100 (where 100 = highest segment).

In [ ]:
radar_data = q(f"""
    SELECT segment_name,
        ROUND(avg_total_spend / MAX(avg_total_spend) OVER() * 100, 0) AS spend_index,
        ROUND(avg_transactions / MAX(avg_transactions) OVER() * 100, 0) AS frequency_index,
        ROUND((1 - avg_recency_days / MAX(avg_recency_days) OVER()) * 100, 0) AS recency_index,
        ROUND(avg_merchants / MAX(avg_merchants) OVER() * 100, 0) AS diversity_index,
        ROUND(avg_active_months / MAX(avg_active_months) OVER() * 100, 0) AS consistency_index
    FROM `{PROJECT}.marts.mart_cluster_profiles`
""")

categories = ['spend_index', 'frequency_index', 'recency_index',
              'diversity_index', 'consistency_index']

fig = go.Figure()
for _, row in radar_data.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row[c] for c in categories] + [row[categories[0]]],
        theta=['Spend', 'Frequency', 'Recency', 'Diversity', 'Consistency', 'Spend'],
        name=row['segment_name'],
        fill='toself', opacity=0.5
    ))

fig.update_layout(title='Segment radar chart (indexed 0-100)',
                  polar=dict(radialaxis=dict(range=[0, 100])),
                  height=500)
fig.show()

### A7. Revenue concentration
"Champions are X% of customers but Y% of revenue" — the pitch headline.

In [ ]:
revenue = q(f"""
    SELECT segment_name,
           COUNT(*) AS customers,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_customers,
           ROUND(SUM(val_trns), 0) AS total_spend,
           ROUND(SUM(val_trns) * 100.0 / SUM(SUM(val_trns)) OVER(), 1) AS pct_revenue
    FROM `{PROJECT}.marts.mart_cluster_output`
    GROUP BY segment_name
    ORDER BY total_spend DESC
""")

fig = go.Figure()
fig.add_trace(go.Bar(name='% of customers', x=revenue['segment_name'],
                     y=revenue['pct_customers'], marker_color='#607D8B'))
fig.add_trace(go.Bar(name='% of revenue', x=revenue['segment_name'],
                     y=revenue['pct_revenue'], marker_color='#2E75B6'))
fig.update_layout(barmode='group', title='Revenue concentration by segment',
                  yaxis_title='%')
fig.show()

champ = revenue[revenue['segment_name'] == 'Champions'].iloc[0]
print(f"\nPitch headline: Champions are {champ['pct_customers']}% of customers "
      f"but drive {champ['pct_revenue']}% of revenue.")

---
## Part B: Churn Prediction Model
---

### B1. Model evaluation — accuracy, precision, recall, F1

In [ ]:
churn_eval = q(f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT}.analytics.churn_classifier`)")

if not churn_eval.empty:
    e = churn_eval.iloc[0]
    print(f"Accuracy:  {e.get('accuracy', 'N/A'):.4f}")
    print(f"Precision: {e.get('precision', 'N/A'):.4f}")
    print(f"Recall:    {e.get('recall', 'N/A'):.4f}")
    print(f"F1 Score:  {e.get('f1_score', 'N/A'):.4f}")
    print(f"Log Loss:  {e.get('log_loss', 'N/A'):.4f}")
    print(f"ROC AUC:   {e.get('roc_auc', 'N/A'):.4f}")

churn_eval.T

### B2. Churn risk distribution
How many customers in each risk level? What's the spend at risk?

In [ ]:
risk_dist = q(f"""
    SELECT churn_risk_level,
           COUNT(*) AS customers,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct,
           ROUND(AVG(churn_probability) * 100, 1) AS avg_churn_pct,
           ROUND(SUM(total_spend), 0) AS total_spend,
           ROUND(AVG(days_since_last), 0) AS avg_days_since_last
    FROM `{PROJECT}.marts.mart_churn_risk`
    GROUP BY churn_risk_level
    ORDER BY avg_churn_pct DESC
""")

churn_colors = {'Critical': '#d32f2f', 'High': '#f57c00',
                'Medium': '#fbc02d', 'Low': '#4caf50', 'Stable': '#2196f3'}

fig = px.pie(risk_dist, values='customers', names='churn_risk_level',
             title='Churn risk distribution (ML-scored)',
             color='churn_risk_level', color_discrete_map=churn_colors)
fig.show()

risk_dist

### B3. Spend at risk by level

In [ ]:
fig = px.bar(risk_dist, x='churn_risk_level', y='total_spend',
             color='churn_risk_level', color_discrete_map=churn_colors,
             title='Spend at risk by churn level')
fig.update_layout(showlegend=False, yaxis_title='Total spend (R)')
fig.show()

# Business headline
critical_high = risk_dist[risk_dist['churn_risk_level'].isin(['Critical', 'High'])]
at_risk_cust = critical_high['customers'].sum()
at_risk_spend = critical_high['total_spend'].sum()
total_cust = risk_dist['customers'].sum()

print(f"\n{at_risk_cust:,} customers ({at_risk_cust/total_cust*100:.1f}%) are Critical or High risk")
print(f"Representing R{at_risk_spend:,.0f} in historical spend")
print(f"A 10% re-engagement rate would recover ~R{at_risk_spend*0.1:,.0f}")

### B4. Churn probability distribution
How are probability scores distributed across all customers?

In [ ]:
prob_dist = q(f"""
    SELECT 
        CASE
            WHEN churn_probability < 0.1 THEN '0-10%'
            WHEN churn_probability < 0.2 THEN '10-20%'
            WHEN churn_probability < 0.3 THEN '20-30%'
            WHEN churn_probability < 0.4 THEN '30-40%'
            WHEN churn_probability < 0.5 THEN '40-50%'
            WHEN churn_probability < 0.6 THEN '50-60%'
            WHEN churn_probability < 0.7 THEN '60-70%'
            WHEN churn_probability < 0.8 THEN '70-80%'
            WHEN churn_probability < 0.9 THEN '80-90%'
            ELSE '90-100%'
        END AS probability_band,
        COUNT(*) AS customers
    FROM `{PROJECT}.marts.mart_churn_risk`
    GROUP BY probability_band
    ORDER BY probability_band
""")

fig = px.bar(prob_dist, x='probability_band', y='customers',
             title='Churn probability distribution',
             color_discrete_sequence=['#2E75B6'])
fig.update_layout(xaxis_title='Churn probability', yaxis_title='Customers')
fig.show()

### B5. Churn risk by demographics
Which customer segments are most at risk?

In [ ]:
churn_demo = q(f"""
    SELECT age_group,
           income_group,
           churn_risk_level,
           COUNT(*) AS customers,
           ROUND(AVG(churn_probability) * 100, 1) AS avg_churn_pct,
           ROUND(SUM(total_spend), 0) AS total_spend
    FROM `{PROJECT}.marts.mart_churn_risk`
    WHERE age_group IS NOT NULL AND income_group IS NOT NULL
    GROUP BY age_group, income_group, churn_risk_level
    ORDER BY avg_churn_pct DESC
""")

# Age group breakdown
age_churn = churn_demo.groupby(['age_group', 'churn_risk_level']).agg(
    customers=('customers', 'sum')
).reset_index()

fig = px.bar(age_churn, x='age_group', y='customers', color='churn_risk_level',
             color_discrete_map=churn_colors, barmode='stack',
             title='Churn risk by age group')
fig.show()

### B6. Win-back targets
High-value customers at high churn risk = best re-engagement targets.

In [ ]:
winback = q(f"""
    SELECT churn_risk_level,
           COUNT(*) AS customers,
           ROUND(AVG(total_spend), 0) AS avg_spend,
           ROUND(SUM(total_spend), 0) AS total_spend_at_risk,
           ROUND(AVG(churn_probability) * 100, 1) AS avg_churn_pct
    FROM `{PROJECT}.marts.mart_churn_risk`
    WHERE churn_risk_level IN ('Critical', 'High')
      AND total_spend > (SELECT AVG(total_spend) FROM `{PROJECT}.marts.mart_churn_risk`)
    GROUP BY churn_risk_level
""")

print('Win-back priority: high-value customers at high churn risk')
print(f'These are above-average spenders who the model flags as likely to leave.\n')
winback